In [1]:
import pandas as pd

from utils.file_utils import get_tickets_as_df

# Data exploration

In [2]:
tickets_df = get_tickets_as_df().dropna(subset=["message", "tags"])
tickets_df["message_created_on"] = pd.to_datetime(tickets_df["message_created_on"])
tickets_df["message"] = tickets_df["message"].astype(str)
tickets_df.head(100)

,ticket_number,ticket_subject,sent_by,ticket_created_on,message_created_on,message,tags
0,1,NaN,Customer,2022-01-18 01:29:01.238861,2022-01-18 01:29:01.238861,"Hello, world!",Testing
1,1,NaN,Customer,2022-01-18 01:29:01.238861,2022-01-18 01:31:08.914884,Is anybody out there?,Testing
2,1,NaN,Agent,2022-01-18 01:29:01.238861,2022-01-18 01:37:46.365888,Ground control to Major Tom,Testing
3,2,NaN,Customer,2022-01-18 04:55:15.792937,2022-01-18 04:55:15.792937,whoa... chat inception,Testing
4,2,NaN,Agent,2022-01-18 04:55:15.792937,2022-01-18 04:55:35.456387,Talking to myself,Testing
...,...,...,...,...,...,...,...
96,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.080650,asd,Testing
97,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.236093,f,Testing
98,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.376098,sdf,Testing
99,31,NaN,Customer,2022-01-31 15:52:37.698712,2022-01-31 15:52:53.522580,ds,Testing


In [53]:
tickets_df.dtypes

ticket_number                  int64
ticket_subject                object
sent_by                       object
ticket_created_on             object
message_created_on    datetime64[ns]
message                       object
tags                          object
dtype: object

In [54]:
# Just investigate the tags, need to check if one ticket has multiple tags
tickets_df.groupby("ticket_number")["tags"].unique()
# conclusion:
# one ticket has multiple tags

ticket_number
1                                            [Testing]
2                                            [Testing]
3                                            [Testing]
4                                            [Testing]
5                                            [Testing]
                             ...                      
1524          [Bug,Infra - Search, Infra - Search,Bug]
1525    [Bug,Ticketing - Email, Ticketing - Email,Bug]
1526                                         [Testing]
1528      [Bug,Customer Profile, Customer Profile,Bug]
1529                                         [Testing]
Name: tags, Length: 1334, dtype: object

In [55]:
# just investigate the tags and the counts
tickets_df["tags"].value_counts()
# conclusion:
# there's a lot of tags with low frequency
# so they are not very useful
# they can be dropped

tags
Testing                                                           1331
Bug                                                                870
Product Question                                                   840
Feature Request                                                    697
Sales                                                              467
                                                                  ... 
Feature Request,Notifications                                        1
Feature Request,Product board                                        1
Product board,Feature Request                                        1
Session Recording - Player,Product Question                          1
Bug,Integrations - Slack,Data - Developer API,Product Question       1
Name: count, Length: 253, dtype: int64

# Feature engineering

In [56]:
cleaned_tickets_df = (
    tickets_df.sort_values("message_created_on")
    .groupby("ticket_number")
    .agg({"message": lambda x: " ".join(x), "tags": "first"})
    .reset_index()
)

cleaned_tickets_df = cleaned_tickets_df[
    cleaned_tickets_df["tags"].str.contains("Spam")
    | cleaned_tickets_df["tags"].str.contains("Bug")
    | cleaned_tickets_df["tags"].str.contains("Product Question")
    | cleaned_tickets_df["tags"].str.contains("Feature Request")
    | cleaned_tickets_df["tags"].str.contains("Sales")
]  # filter out spam, bug, product question, feature request, sales


cleaned_tickets_df["tags"].value_counts()
# we still can see a lot of tags with low frequency

tags
Product Question                               128
Bug                                            126
Sales                                          117
Feature Request                                101
Spam                                            22
                                              ... 
Feature Request,Compliance - HIPAA               1
Product Question,Chatbot                         1
Feature Request,Ticketing - Delivery status      1
Onboarding,Product Question                      1
Bug,Infra - Search                               1
Name: count, Length: 146, dtype: int64

In [57]:
def map_categories(cat: str) -> str:
    if "Bug" in cat:
        return "Bug"
    elif "Product Question" in cat:
        return "Product Question"
    elif "Feature Request" in cat:
        return "Feature Request"
    elif "Sales" in cat:
        return "Sales"
    elif "Spam" in cat:
        return "Spam"
    else:
        return "Other"


cleaned_tickets_df["tags"] = cleaned_tickets_df["tags"].apply(map_categories)
cleaned_tickets_df["tags"].value_counts()
# we can see that we only have 5 categories
# and they are: Bug, Product Question, Feature Request, Sales, Spam



tags
Feature Request     212
Product Question    201
Bug                 193
Sales               120
Spam                 23
Name: count, dtype: int64

In [58]:
cleaned_tickets_df

,ticket_number,message,tags
20,21,Hi! What is your pricing? Can I talk with some...,Sales
21,22,"Hi, how much is Atlas? Looks like it, yeah.",Sales
29,30,hey not super urgent but fyi in ramp bank you ...,Feature Request
38,39,some minor feedback - i seem to need to sign i...,Feature Request
41,42,Hello! Just came across your site: do you offe...,Feature Request
...,...,...,...
1327,1522,"Good morning, everyone. A customer contacted u...",Bug
1328,1523,Hey quick question - we are figuring out how t...,Feature Request
1329,1524,Hi - I'm running into an issue where there are...,Bug
1330,1525,Hey @Rahul Asati!It looks like we are still ge...,Bug


# Save to csv

In [59]:
cleaned_tickets_df.to_csv("data/cleaned_tickets_v4.csv", index=False)